# 02 — Feature Engineering and Model Development

Chapter 3, Sections 3.6.1 (repayment history), 3.5.5 (encoding and scaling),
3.9 (partitioning, classifiers, tuning, calibration).

**Run order matters.** Prior-loan features are built and merged BEFORE the split,
and every transformation is fitted on the training set only.

Outputs saved for notebook 03:
- `artifacts/splits.joblib` — the three sets and the audit columns
- `artifacts/tuned_models.joblib` — fitted, calibrated models
- `artifacts/study_summary.csv` — Optuna best parameters per model

## 1. Setup

In [1]:
import os, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore', category=FutureWarning)

DATA = '../project/'
ART  = 'artifacts'
os.makedirs(ART, exist_ok=True)

SEED      = 42
N_TRIALS  = 50          # Optuna trials per model (Section 3.9.3)
CV_FOLDS  = 5
TUNE_OBJ  = 'neg_log_loss'   # threshold-free; see 3.9.3

np.random.seed(SEED)
print('seed', SEED, '| trials', N_TRIALS, '| objective', TUNE_OBJ)

seed 42 | trials 50 | objective neg_log_loss


## 2. Rebuild the base table

This repeats notebook 01's merge, cleaning and derived features so that this
notebook runs standalone. The figures printed here must match notebook 01.

In [2]:
perf = pd.read_csv(DATA + 'trainperf.csv')
demo = pd.read_csv(DATA + 'traindemographics.csv')
prev = pd.read_csv(DATA + 'trainprevloans.csv')

demo_u = demo.drop_duplicates(subset='customerid', keep='first')
df = perf.merge(demo_u, on='customerid', how='left', indicator=True)
df['has_demographics'] = (df['_merge'] == 'both').astype(int)
df = df.drop(columns='_merge')

for c in ['approveddate', 'creationdate', 'birthdate']:
    df[c] = pd.to_datetime(df[c], errors='coerce')
    try:
        df[c] = df[c].dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

print('rows:', len(df), '| with demographics:', df['has_demographics'].sum())

rows: 4368 | with demographics: 3269


In [3]:
# Age at approval + bands (Section 3.6.2)
df['age_at_approval'] = (df['approveddate'] - df['birthdate']).dt.days / 365.25
implausible = df['age_at_approval'].notna() & (
    (df['age_at_approval'] < 18) | (df['age_at_approval'] > 80))
df.loc[implausible, 'age_at_approval'] = pd.NA

df['age_band'] = pd.cut(
    df['age_at_approval'].astype('float'),
    bins=[18, 25, 35, 45, 80],
    labels=['18-24', '25-34', '35-44', '45+'],
    right=False,
).astype('object').fillna('Unknown')
df['age_band'] = df['age_band'].replace({'45+': '35+', '35-44': '35+'})

# Interest margin (Equation 3.1)
df['interest_margin'] = (df['totaldue'] - df['loanamount']) / df['loanamount']

print('implausible ages:', implausible.sum())
print(df['age_band'].value_counts().to_string())

implausible ages: 0
age_band
25-34      1941
Unknown    1099
35+        1064
18-24       264


In [4]:
# Region from GPS (Section 3.6.2)
import geopandas as gpd, re

states = gpd.read_file('gadm41_NGA_1.json')[['NAME_1', 'geometry']]
zones = {
    'North Central': ['Benue', 'Kogi', 'Kwara', 'Nasarawa', 'Niger', 'Plateau',
                      'Federal Capital Territory'],
    'North East':    ['Adamawa', 'Bauchi', 'Borno', 'Gombe', 'Taraba', 'Yobe'],
    'North West':    ['Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Sokoto', 'Zamfara'],
    'South East':    ['Abia', 'Anambra', 'Ebonyi', 'Enugu', 'Imo'],
    'South South':   ['Akwa Ibom', 'Bayelsa', 'Cross River', 'Delta', 'Edo', 'Rivers'],
    'South West':    ['Ekiti', 'Lagos', 'Ogun', 'Ondo', 'Osun', 'Oyo'],
}
norm = lambda s: re.sub(r'[^a-z]', '', s.lower()).replace('nassarawa', 'nasarawa')
state_to_zone = {norm(s): z for z, lst in zones.items() for s in lst}
states['zone'] = states['NAME_1'].map(lambda s: state_to_zone.get(norm(s)))
assert states['zone'].notna().all()

has_gps = df['longitude_gps'].notna() & df['latitude_gps'].notna()
pts = gpd.GeoDataFrame(
    df.loc[has_gps, []],
    geometry=gpd.points_from_xy(df.loc[has_gps, 'longitude_gps'],
                                df.loc[has_gps, 'latitude_gps']),
    crs='EPSG:4326').to_crs(states.crs)

joined = gpd.sjoin(pts, states[['NAME_1', 'zone', 'geometry']],
                   how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]

df['state']  = joined['NAME_1'].reindex(df.index)
df['region'] = joined['zone'].reindex(df.index).fillna('Unknown')
df['region_fair'] = df['region'].replace(
    {z: 'Other regions' for z in ['North West', 'South East', 'North East']})

print('outside Nigeria:', (has_gps & df['state'].isna()).sum())
print(df['region_fair'].value_counts().to_string())

outside Nigeria: 36
region_fair
South West       2145
Unknown          1135
North Central     511
South South       366
Other regions     211


## 3. Section 3.6.1 — repayment history features

The previous-loans file is aggregated to one row per customer. **Only prior
loans approved strictly before the current loan's approval date are included**,
so no feature uses information unavailable at the time of the decision.

In [5]:
for c in ['approveddate', 'firstduedate', 'firstrepaiddate', 'closeddate']:
    prev[c] = pd.to_datetime(prev[c], errors='coerce')
    try:
        prev[c] = prev[c].dt.tz_localize(None)
    except (TypeError, AttributeError):
        pass

prev['days_late'] = (prev['firstrepaiddate'] - prev['firstduedate']).dt.days
prev['was_late']  = (prev['days_late'] > 0).astype(int)

print('prior loans:', len(prev), '| customers:', prev['customerid'].nunique())
print('overall late rate:', round(prev['was_late'].mean(), 4))

prior loans: 18183 | customers: 4359
overall late rate: 0.1738


In [6]:
# LEAKAGE FILTER: attach each customer's current approval date, keep only
# prior loans approved strictly before it.
cur = df[['customerid', 'approveddate']].rename(
    columns={'approveddate': 'current_approved'})

hist = prev.merge(cur, on='customerid', how='inner')
before = hist['approveddate'] < hist['current_approved']

print(f'prior-loan rows total      : {len(hist)}')
print(f'dropped (on/after current) : {(~before).sum()}')
hist = hist[before].copy()
print(f'retained                   : {len(hist)}')

prior-loan rows total      : 18183
dropped (on/after current) : 0
retained                   : 18183


In [7]:
# Aggregate to one row per customer (Table 3.3)
agg = hist.groupby('customerid').agg(
    prev_loan_count     = ('systemloanid', 'count'),
    prev_amount_mean    = ('loanamount', 'mean'),
    prev_amount_max     = ('loanamount', 'max'),
    prev_late_rate      = ('was_late', 'mean'),
    prev_days_late_mean = ('days_late', 'mean'),
    prev_termdays_mean  = ('termdays', 'mean'),
    last_prev_approved  = ('approveddate', 'max'),
).reset_index()

df = df.merge(agg, on='customerid', how='left')

# Days between the last prior loan and this one
df['days_since_last_loan'] = (
    df['approveddate'] - df['last_prev_approved']).dt.days
df = df.drop(columns='last_prev_approved')

# Customers with no usable history
df['has_history']     = df['prev_loan_count'].notna().astype(int)
df['prev_loan_count'] = df['prev_loan_count'].fillna(0)

print('customers with no prior loans:',
      (df['has_history'] == 0).sum(),
      f"({(df['has_history'] == 0).mean():.2%})")
print()
print(df[['prev_loan_count', 'prev_amount_mean', 'prev_late_rate',
          'prev_days_late_mean', 'days_since_last_loan']].describe().round(2).to_string())

customers with no prior loans: 9 (0.21%)

       prev_loan_count  prev_amount_mean  prev_late_rate  prev_days_late_mean  days_since_last_loan
count          4368.00           4359.00         4359.00              4359.00               4359.00
mean              4.16          12909.51            0.18                -2.45                 38.99
std               3.66           4819.32            0.30                 7.41                 46.65
min               0.00           5000.00            0.00               -32.00                  0.00
25%               1.00          10000.00            0.00                -5.50                 24.00
50%               3.00          10000.00            0.00                -2.40                 29.00
75%               6.00          14285.71            0.29                 0.00                 36.00
max              26.00          36666.67            1.00               121.00                544.00


In [8]:
# Feature cleanup before the split
# referredby is a referrer ID, not a category — one-hot encoding it produced
# hundreds of near-empty columns. Reduce it to whether a referrer existed.
if 'referredby' in df.columns:
    df['was_referred'] = df['referredby'].notna().astype(int)
    df = df.drop(columns='referredby')

# bank_branch_clients: 99.24% missing and high-cardinality where present
if 'bank_branch_clients' in df.columns:
    print('bank_branch_clients distinct values:', df['bank_branch_clients'].nunique())
    df = df.drop(columns='bank_branch_clients')

# totaldue is fully redundant given loanamount and interest_margin
df = df.drop(columns='totaldue', errors='ignore')

print('\nwas_referred:')
print(df['was_referred'].value_counts().to_string())
print('\nremaining categorical columns:')
for c in df.select_dtypes(include='object').columns:
    print(f'  {c:<30} {df[c].nunique()}')

bank_branch_clients distinct values: 31

was_referred:
was_referred
0    3781
1     587

remaining categorical columns:
  customerid                     4368
  good_bad_flag                  2
  bank_account_type              3
  bank_name_clients              18
  employment_status_clients      6
  level_of_education_clients     4
  age_band                       4
  state                          36
  region                         7
  region_fair                    5


## 4. Section 3.9.1 — train / validation / test split

Every engineered feature now exists, so the split can be taken.

In [9]:
from sklearn.model_selection import train_test_split

TARGET = 'good_bad_flag'

drop_cols = [TARGET, 'customerid', 'systemloanid', 'loannumber',
             'approveddate', 'creationdate', 'birthdate',
             'longitude_gps', 'latitude_gps', 'state',
             'region_fair', 'age_band']
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols)
y = (df[TARGET] == 'Bad').astype(int)

required = ['age_at_approval', 'interest_margin', 'region', 'has_demographics',
            'loanamount', 'termdays', 'prev_loan_count', 'prev_late_rate']
missing = [c for c in required if c not in X.columns]
assert not missing, f'Missing features — re-run cells above: {missing}'

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

for name, yy in [('Train', y_tr), ('Validation', y_val), ('Test', y_test)]:
    print(f'{name:<11} n = {len(yy):>5}   default rate = {yy.mean():.4f}')

print(f'\nFeatures ({len(X.columns)}):')
print(sorted(X.columns.tolist()))

Train       n =  2620   default rate = 0.2179
Validation  n =   874   default rate = 0.2185
Test        n =   874   default rate = 0.2174

Features (19):
['age_at_approval', 'bank_account_type', 'bank_name_clients', 'days_since_last_loan', 'employment_status_clients', 'has_demographics', 'has_history', 'interest_margin', 'level_of_education_clients', 'loanamount', 'prev_amount_max', 'prev_amount_mean', 'prev_days_late_mean', 'prev_late_rate', 'prev_loan_count', 'prev_termdays_mean', 'region', 'termdays', 'was_referred']


In [10]:
# Protected-attribute group sizes in the test set (for Chapter 4)
print('region_fair (test)')
print(df.loc[X_test.index, 'region_fair'].value_counts().to_string())
print()
print('age_band (test)')
print(df.loc[X_test.index, 'age_band'].value_counts().to_string())

region_fair (test)
region_fair
South West       434
Unknown          233
North Central     92
South South       77
Other regions     38

age_band (test)
age_band
25-34      387
Unknown    228
35+        218
18-24       41


## 5. Section 3.5.5 — encoding and scaling

Two preprocessing pipelines are built:

- **`pre_ohe`** — median imputation + standardisation for numeric columns,
  "Unknown" imputation + one-hot encoding for categoricals. Used by Logistic
  Regression, Random Forest and XGBoost.
- **`pre_native`** — median imputation only, categoricals left as `category`
  dtype. Used by LightGBM and CatBoost, which handle categories natively.

Both are fitted on the training set inside the cross-validation loop, so no
validation or test information reaches the transformations.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print('numeric     :', num_cols)
print()
print('categorical :', cat_cols)
for c in cat_cols:
    print(f'   {c:<28} {X[c].nunique()} categories')

numeric     : ['loanamount', 'termdays', 'has_demographics', 'age_at_approval', 'interest_margin', 'prev_loan_count', 'prev_amount_mean', 'prev_amount_max', 'prev_late_rate', 'prev_days_late_mean', 'prev_termdays_mean', 'days_since_last_loan', 'has_history', 'was_referred']

categorical : ['bank_account_type', 'bank_name_clients', 'employment_status_clients', 'level_of_education_clients', 'region']
   bank_account_type            3 categories
   bank_name_clients            18 categories
   employment_status_clients    6 categories
   level_of_education_clients   4 categories
   region                       7 categories


In [12]:
def make_ohe_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler()),
        ]), num_cols),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('ohe',    OneHotEncoder(handle_unknown='ignore',
                                     sparse_output=False)),
        ]), cat_cols),
    ], remainder='drop')


def make_native_frame(frame):
    """Median-impute numerics, cast categoricals to pandas 'category'."""
    out = frame.copy()
    for c in cat_cols:
        out[c] = out[c].fillna('Unknown').astype('category')
    return out


# Category levels must be identical across the three sets, or LightGBM and
# CatBoost will encode them differently.
cat_levels = {c: sorted(X[c].fillna('Unknown').astype(str).unique())
              for c in cat_cols}

def make_native(frame):
    out = frame.copy()
    for c in cat_cols:
        out[c] = pd.Categorical(out[c].fillna('Unknown').astype(str),
                                categories=cat_levels[c])
    return out

Xn_tr, Xn_val, Xn_test = map(make_native, [X_tr, X_val, X_test])

_probe = make_ohe_preprocessor().fit(X_tr)
print('one-hot encoded feature count:', _probe.transform(X_tr).shape[1])
print('native-categorical columns   :', len(cat_cols))

one-hot encoded feature count: 55
native-categorical columns   : 5


## 6. Section 3.9.3 — hyperparameter tuning

Optuna with a TPE sampler, stratified 5-fold cross-validation on the training
set only. The objective is log loss — threshold-free, and it rewards
probabilities being numerically correct rather than merely well ranked, which
matters because the cost-derived threshold in Section 3.8.3 assumes calibrated
probabilities.

Search spaces follow Table 3.5.

In [13]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)


def score_ohe(estimator, Xd, yd):
    pipe = Pipeline([('pre', make_ohe_preprocessor()), ('clf', estimator)])
    return cross_val_score(pipe, Xd, yd, cv=cv, scoring=TUNE_OBJ, n_jobs=-1).mean()


def score_native(estimator, Xd, yd):
    return cross_val_score(estimator, Xd, yd, cv=cv, scoring=TUNE_OBJ,
                           n_jobs=1).mean()

In [14]:
from sklearn.metrics import log_loss

# CatBoost wants plain strings for categorical columns, and it cannot be
# cloned by scikit-learn, so it gets its own frames and its own CV loop.
def make_catframe(frame):
    out = frame.copy()
    for c in cat_cols:
        out[c] = out[c].astype(str).replace({'nan': 'Unknown'})
    return out

Xc_tr, Xc_val, Xc_test = map(make_catframe, [X_tr, X_val, X_test])


def score_cat(params):
    """Manual CV for CatBoost, which scikit-learn cannot clone."""
    scores = []
    for tr_idx, va_idx in cv.split(Xc_tr, y_tr):
        clf = CatBoostClassifier(cat_features=cat_cols, verbose=0,
                                 random_seed=SEED, allow_writing_files=False,
                                 thread_count=-1,
                                 **params)
        clf.fit(Xc_tr.iloc[tr_idx], y_tr.iloc[tr_idx])
        p = clf.predict_proba(Xc_tr.iloc[va_idx])[:, 1]
        scores.append(-log_loss(y_tr.iloc[va_idx], p))
    return float(np.mean(scores))


print(Xc_tr.shape, '|', Xc_tr[cat_cols].dtypes.unique())
print(Xc_tr[cat_cols[0]].value_counts().head().to_string())

(2620, 19) | [dtype('O')]
bank_account_type
Savings    1552
Unknown     669
Other       376
Current      23


In [15]:
def obj_lr(trial):
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    C       = trial.suggest_float('C', 1e-3, 100, log=True)
    clf = LogisticRegression(penalty=penalty, C=C, solver='liblinear',
                             max_iter=2000, random_state=SEED)
    return score_ohe(clf, X_tr, y_tr)


def obj_rf(trial):
    clf = RandomForestClassifier(
        n_estimators     = trial.suggest_int('n_estimators', 100, 1000, step=50),
        max_depth        = trial.suggest_int('max_depth', 3, 20),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50),
        n_jobs=-1, random_state=SEED)
    return score_ohe(clf, X_tr, y_tr)


def obj_xgb(trial):
    clf = XGBClassifier(
        n_estimators  = trial.suggest_int('n_estimators', 100, 1000, step=50),
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        max_depth     = trial.suggest_int('max_depth', 3, 10),
        subsample     = trial.suggest_float('subsample', 0.5, 1.0),
        eval_metric='logloss', tree_method='hist',
        n_jobs=-1, random_state=SEED)
    return score_ohe(clf, X_tr, y_tr)


def obj_lgbm(trial):
    clf = LGBMClassifier(
        n_estimators      = trial.suggest_int('n_estimators', 100, 1000, step=50),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 15, 255),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        verbose=-1, n_jobs=-1, random_state=SEED)
    return score_native(clf, Xn_tr, y_tr)


def obj_cat(trial):
    params = {
        'iterations':    trial.suggest_int('iterations', 100, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':         trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':   trial.suggest_float('l2_leaf_reg', 1, 10),
    }
    return score_cat(params)


OBJECTIVES = {'LR': obj_lr, 'RF': obj_rf, 'XGBoost': obj_xgb,
              'LightGBM': obj_lgbm, 'CatBoost': obj_cat}
print('objectives defined:', list(OBJECTIVES))

objectives defined: ['LR', 'RF', 'XGBoost', 'LightGBM', 'CatBoost']


In [16]:
# SLOW CELL — roughly 30-90 minutes for all five models.
# Results are cached; delete artifacts/studies.joblib to force a re-run.
STUDY_PATH = f'{ART}/studies.joblib'

if os.path.exists(STUDY_PATH):
    best_params = joblib.load(STUDY_PATH)
    print('loaded cached tuning results')
else:
    best_params = {}
    for name, objective in OBJECTIVES.items():
        t0 = time.time()
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        best_params[name] = study.best_params
        print(f'{name:<10} best {TUNE_OBJ} = {study.best_value:.5f} '
              f'({time.time() - t0:.0f}s)')
    joblib.dump(best_params, STUDY_PATH)

for name, p in best_params.items():
    print(f'\n{name}: {p}')

LR         best neg_log_loss = -0.48299 (17s)
RF         best neg_log_loss = -0.47366 (108s)
XGBoost    best neg_log_loss = -0.47178 (49s)
LightGBM   best neg_log_loss = -0.47415 (85s)
CatBoost   best neg_log_loss = -0.47444 (5326s)

LR: {'penalty': 'l2', 'C': 0.05370325013839185}

RF: {'n_estimators': 850, 'max_depth': 14, 'min_samples_leaf': 10}

XGBoost: {'n_estimators': 200, 'learning_rate': 0.023874108472166155, 'max_depth': 3, 'subsample': 0.7827947709396337}

LightGBM: {'n_estimators': 100, 'learning_rate': 0.02180147370879372, 'num_leaves': 177, 'min_child_samples': 85}

CatBoost: {'iterations': 400, 'learning_rate': 0.012561081531777471, 'depth': 9, 'l2_leaf_reg': 7.757654535587468}


In [17]:
# Selected values against their search ranges — flag any at a boundary
RANGES = {
    'LR':       {'C': (1e-3, 100)},
    'RF':       {'n_estimators': (100, 1000), 'max_depth': (3, 20),
                 'min_samples_leaf': (1, 50)},
    'XGBoost':  {'n_estimators': (100, 1000), 'learning_rate': (0.01, 0.3),
                 'max_depth': (3, 10), 'subsample': (0.5, 1.0)},
    'LightGBM': {'n_estimators': (100, 1000), 'learning_rate': (0.01, 0.3),
                 'num_leaves': (15, 255), 'min_child_samples': (5, 100)},
    'CatBoost': {'iterations': (100, 1000), 'learning_rate': (0.01, 0.3),
                 'depth': (4, 10), 'l2_leaf_reg': (1, 10)},
}

rows = []
for model, params in best_params.items():
    for k, v in params.items():
        lo_hi = RANGES.get(model, {}).get(k)
        flag = ''
        if lo_hi:
            lo, hi = lo_hi
            span = hi - lo
            if v >= hi - 0.05 * span:
                flag = 'AT UPPER BOUND'
            elif v <= lo + 0.05 * span:
                flag = 'at lower bound'
        rows.append({'Model': model, 'Parameter': k, 'Selected': v,
                     'Range': f'{lo_hi[0]}-{lo_hi[1]}' if lo_hi else '-',
                     'Note': flag})

summary = pd.DataFrame(rows)
summary.to_csv(f'{ART}/study_summary.csv', index=False)
print(summary.to_string(index=False))

   Model         Parameter  Selected     Range           Note
      LR           penalty        l2         -               
      LR                 C  0.053703 0.001-100 at lower bound
      RF      n_estimators       850  100-1000               
      RF         max_depth        14      3-20               
      RF  min_samples_leaf        10      1-50               
 XGBoost      n_estimators       200  100-1000               
 XGBoost     learning_rate  0.023874  0.01-0.3 at lower bound
 XGBoost         max_depth         3      3-10 at lower bound
 XGBoost         subsample  0.782795   0.5-1.0               
LightGBM      n_estimators       100  100-1000 at lower bound
LightGBM     learning_rate  0.021801  0.01-0.3 at lower bound
LightGBM        num_leaves       177    15-255               
LightGBM min_child_samples        85     5-100               
CatBoost        iterations       400  100-1000               
CatBoost     learning_rate  0.012561  0.01-0.3 at lower bound
CatBoost

## 7. Section 3.9.4 — probability calibration

Each tuned model is refitted on the training set and calibrated on the
**validation set** using `CalibratedClassifierCV` with `cv='prefit'`. Both
Platt scaling (sigmoid) and isotonic regression are tried; the one with the
lower Brier score on the validation set is kept.

Brier scores before and after calibration are recorded for Chapter 4.

In [18]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score

try:
    from sklearn.frozen import FrozenEstimator
    def freeze(est): return FrozenEstimator(est)
    FROZEN_KW = {}
except ImportError:
    def freeze(est): return est
    FROZEN_KW = {'cv': 'prefit'}



def build(name):
    p = best_params[name]
    if name == 'LR':
        return Pipeline([('pre', make_ohe_preprocessor()),
                         ('clf', LogisticRegression(solver='liblinear',
                                                    max_iter=2000,
                                                    random_state=SEED, **p))]), False
    if name == 'RF':
        return Pipeline([('pre', make_ohe_preprocessor()),
                         ('clf', RandomForestClassifier(n_jobs=-1,
                                                        random_state=SEED, **p))]), False
    if name == 'XGBoost':
        return Pipeline([('pre', make_ohe_preprocessor()),
                         ('clf', XGBClassifier(eval_metric='logloss',
                                               tree_method='hist', n_jobs=-1,
                                               random_state=SEED, **p))]), False
    if name == 'LightGBM':
        return LGBMClassifier(verbose=-1, n_jobs=-1, random_state=SEED, **p), True
    if name == 'CatBoost':
        return CatBoostClassifier(cat_features=cat_cols, verbose=0,
                                  random_seed=SEED, allow_writing_files=False,
                                  thread_count=-1,
                                  **p), 'cat'
    raise ValueError(name)


models, calib_log = {}, []

for name in OBJECTIVES:
    est, native = build(name)
    Xa, Xb = (Xn_tr, Xn_val) if native else (X_tr, X_val)

    est.fit(Xa, y_tr)
    p_raw = est.predict_proba(Xb)[:, 1]
    brier_raw = brier_score_loss(y_val, p_raw)

    best = None
    for method in ['sigmoid', 'isotonic']:
        cal = CalibratedClassifierCV(FrozenEstimator(est), method=method)
        cal.fit(Xb, y_val)
        b = brier_score_loss(y_val, cal.predict_proba(Xb)[:, 1])
        if best is None or b < best[1]:
            best = (method, b, cal)

    method, brier_cal, cal = best
    models[name] = {'model': cal, 'native': native, 'method': method}
    calib_log.append({'Model': name, 'Method': method,
                      'Brier (raw)': round(brier_raw, 5),
                      'Brier (calibrated)': round(brier_cal, 5),
                      'AUC (val)': round(roc_auc_score(y_val, p_raw), 4)})
    print(f'{name:<10} {method:<9} Brier {brier_raw:.5f} -> {brier_cal:.5f}')

calib_df = pd.DataFrame(calib_log)
calib_df.to_csv(f'{ART}/calibration_summary.csv', index=False)
print()
print(calib_df.to_string(index=False))

LR         isotonic  Brier 0.14927 -> 0.14484
RF         isotonic  Brier 0.14997 -> 0.14375
XGBoost    isotonic  Brier 0.15211 -> 0.14853
LightGBM   isotonic  Brier 0.15305 -> 0.14831
CatBoost   isotonic  Brier 0.14884 -> 0.14403

   Model   Method  Brier (raw)  Brier (calibrated)  AUC (val)
      LR isotonic      0.14927             0.14484     0.7137
      RF isotonic      0.14997             0.14375     0.7117
 XGBoost isotonic      0.15211             0.14853     0.6970
LightGBM isotonic      0.15305             0.14831     0.6857
CatBoost isotonic      0.14884             0.14403     0.7239


Calibrating on the validation set and then selecting the calibration method
on that same set is mildly optimistic. It is acceptable here because the test
set remains untouched, but the point is worth a sentence in Chapter 4.

## 8. Save artifacts for notebook 03

In [19]:
audit = df.loc[:, ['region_fair', 'age_band', 'region',
                   'loanamount', 'interest_margin']]

bundle = {
    'X_tr': X_tr, 'X_val': X_val, 'X_test': X_test,
    'Xn_tr': Xn_tr, 'Xn_val': Xn_val, 'Xn_test': Xn_test,
    'y_tr': y_tr, 'y_val': y_val, 'y_test': y_test,
    'num_cols': num_cols, 'cat_cols': cat_cols,
    'audit': audit,
    'seed': SEED,
}
joblib.dump(bundle, f'{ART}/splits.joblib')
joblib.dump(models, f'{ART}/tuned_models.joblib')

print('saved:')
for f in sorted(os.listdir(ART)):
    size = os.path.getsize(os.path.join(ART, f)) / 1024
    print(f'  {f:<28} {size:>8.1f} KB')

saved:
  calibration_summary.csv           0.2 KB
  splits.joblib                  1427.1 KB
  studies.joblib                    0.3 KB
  study_summary.csv                 0.7 KB
  tuned_models.joblib           13866.2 KB


## 9. Quick sanity check

A first look at validation performance. The full evaluation, thresholds, costs,
SHAP and the fairness audit are in notebook 03.

In [20]:
rows = []
for name, m in models.items():
    Xb = Xn_val if m['native'] else X_val
    p = m['model'].predict_proba(Xb)[:, 1]
    rows.append({
        'Model': name,
        'AUC': round(roc_auc_score(y_val, p), 4),
        'Brier': round(brier_score_loss(y_val, p), 5),
        'mean p': round(p.mean(), 4),
        'p >= 0.1837': int((p >= 0.1837).sum()),
    })

res = pd.DataFrame(rows).sort_values('AUC', ascending=False)
print(res.to_string(index=False))
print(f'\nActual default rate in validation: {y_val.mean():.4f}')
print('Mean predicted probability should sit close to this if calibrated.')

   Model    AUC   Brier  mean p  p >= 0.1837
CatBoost 0.7372 0.14403  0.2185          494
      LR 0.7283 0.14484  0.2185          520
      RF 0.7269 0.14375  0.2185          383
 XGBoost 0.7100 0.14853  0.2185          307
LightGBM 0.7047 0.14831  0.2185          276

Actual default rate in validation: 0.2185
Mean predicted probability should sit close to this if calibrated.


In [22]:
line = lambda t: print('\n' + '=' * 60 + f'\n{t}\n' + '=' * 60)
line('8. BASELINE COMPARISON')
from sklearn.metrics import log_loss

_frames = {'ohe': X_val, 'lgbm': Xn_val, 'cat': Xc_val,
           True: Xn_val, False: X_val}

base = np.full(len(y_val), y_tr.mean())
print('base-rate log loss :', round(log_loss(y_val, base), 5))

for name, m in models.items():
    key = m.get('kind', m.get('native'))
    Xb = _frames[key]
    p = m['model'].predict_proba(Xb)[:, 1]
    print(f'{name:<10} log loss = {log_loss(y_val, p):.5f}')


8. BASELINE COMPARISON
base-rate log loss : 0.52505
LR         log loss = 0.45411
RF         log loss = 0.45387
XGBoost    log loss = 0.46536
LightGBM   log loss = 0.46586
CatBoost   log loss = 0.44967
